---

# Mapa de Mann-Kendall para focos de calor 2003-2025

---

- `OBJETIVO`:
> Este código acumula os flashes para uma região e dia específico numa grade de 8km X 8km e salva num arquivo NETCDF. As ocorrências de flashes também são salvas  num dataframe contendo o tempo (tempo do primeiro evento do flash), latitude e longitude do flash.



- `DADOS DE ENTRADA`:
> Dados do sensor GLM do satélite [GOES-16](https://noaa-goes16.s3.amazonaws.com/index.html#GLM-L2-LCFA/) ou [GOES-19](https://noaa-goes19.s3.amazonaws.com/index.html#GLM-L2-LCFA/) fornecido pela AMAZON. Exemplo de nome do arquivo: `OR_GLM-L2-LCFA_G16_s20201820000000_e20201820000200_c20201820000224.nc`


- `DADOS DE SAÍDA`:
> 1. Arquivo NETCDF de flashes numa grade de 8km X 8km. Exemplo: `flash_glm_goes_2020-06-30.nc`
> 2. Dataframe do tempo, latitude e longitude do flash. Exemplo: `flash_glm_goes_2020-06-30.csv`


- `OBSERVAÇÕES`:
   > Tempo de processamento de 1 dia de dados = `1h21min43s`

- `REALIZADO POR`:
> Enrique V. Mattos - 29/05/2026

- `ATUALIZADO POR`:
> Enrique V. Mattos - 29/05/2026
---


In [ ]:
#=========================================================================================================================#
#                                     INSTALAÇÃO E IMPORTAÇÃO DAS BIBLIOTECAS
#=========================================================================================================================#
# instalações
!pip install -q ultraplot cartopy salem rasterio pyproj geopandas pymannkendall
import ultraplot as uplt
import cartopy.crs as ccrs
import cartopy.io.shapereader as shpreader
import pandas as pd
from datetime import timedelta, datetime
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
import os
import imageio
import glob
import calendar
import pandas as pd
import glob
import numpy as np
import xarray as xr
import time
import salem
import pymannkendall as mk
import matplotlib.ticker as mticker
from matplotlib.ticker import FormatStrFormatter
import warnings
warnings.filterwarnings("ignore")

#=========================================================================================================================#
#                                        MONTA O GOOGLE DRIVE E CRIA OS DIRETÓRIOS
#=========================================================================================================================#
# monta o drive
from google.colab import drive
drive.mount('/content/drive')

# diretório raiz
dir = '/content/drive/MyDrive/PYHTON/00_GITHUB/000_CODIGOS_REFERENCIA/07_TESTE_MANN_KENDALL'

# diretório de entrada
dir_input = f'{dir}/output/02_netcdf_focos_por_ano'

# diretório de saída
dir_output = f'{dir}/output/figuras'

# cria pasta de saída
os.makedirs(dir_output, exist_ok=True)

#=========================================================================================================================#
#                                              CARREGA SHAPEFILE DO ESTADO DE SP
#=========================================================================================================================#
# leitura do shapefile de SP
shapefile_sp = salem.read_shapefile('https://github.com/evmpython/shapefile/raw/main/UFs/SP/SP_UF_2019.shp')

# limites de SP
lonmin_SP, lonmax_SP, latmin_SP, latmax_SP = -53.3-0.5, -43.9+0.5, -25.4-0.5, -19.7+0.5

#=========================================================================================================================#
#                                      LEITURA DO ARQUIVO NETCDF DE FOCOS DE CALOR
#=========================================================================================================================#
# mostrando os dados de focos
focos_por_ano_2003_a_2025 = xr.open_dataset(f'{dir_input}/focos_anuais_brasil_20km_AQUA_2003_2025.nc')

# limites do estado de SP
lonmin_sp, lonmax_sp, latmin_sp, latmax_sp = -53.3, -43.9, -25.4, -19.7

# recorta os dados para o Estado de SP
focos_por_ano_2003_a_2025_sp = focos_por_ano_2003_a_2025.sel(lat=slice(latmax_sp, latmin_sp), lon=slice(lonmin_sp, lonmax_sp)).sel(time=slice('2003', '2024'))

#=========================================================================================================================#
#                                                   CALCULA MANN-KENDALL
#=========================================================================================================================#
def mann_kendall_test_wrapper(time_series_data):

    # Garante que os dados não são todos NaN e que há pelo menos 3 pontos não-NaN

    data_non_nan = time_series_data[~np.isnan(time_series_data)]
    if data_non_nan.size < 3:
        return np.nan, np.nan

    # Executa o teste original de Mann-Kendall
    result = mk.original_test(data_non_nan)

    # Retorna o slope (inclinação) e o p-value (valor-p) do teste
    return result.slope, result.p

# Aplica a função universal (Ufunc) ao DataArray 'focos'
# Isso permite aplicar a função 'mann_kendall_test_wrapper' a cada série temporal (cada ponto de grade) de forma eficiente

slope_p_values = xr.apply_ufunc(mann_kendall_test_wrapper,
                                focos_por_ano_2003_a_2025_sp['focos'],
                                input_core_dims=[['time']],            # Indica que a função opera ao longo da dimensão 'time' de cada série
                                output_core_dims=[[], []],             # A saída são dois valores escalares (slope e p-value) para cada ponto de grade
                                exclude_dims=set(('time',)),           # A dimensão 'time' é consumida pela função interna
                                vectorize=True,                        # Importante para aplicar uma função de 1D em "fatias" de arrays multidimensionais
                                dask='parallelized',                   # Habilita o Dask para computação paralela, se o xarray estiver configurado para isso
                                output_dtypes=[np.float64, np.float64] # Define explicitamente os tipos de dados de saída
                                )

# Desempacota os resultados em DataArrays separados para 'slope' e 'p_value'
slope = slope_p_values[0]
p_value = slope_p_values[1]

# Atribui nomes significativos aos DataArrays resultantes
slope.name = 'slope'
p_value.name = 'p_value'

# Adiciona atributos 'long_name' e 'units' para descrição no NetCDF
slope.attrs['long_name'] = 'Inclinação (Slope) do Teste de Mann-Kendall'
slope.attrs['units'] = 'focos por ano'
p_value.attrs['long_name'] = 'Valor-P (P-value) do Teste de Mann-Kendall'
p_value.attrs['units'] = 'adimensional'

# Cria um novo Dataset com os resultados do Mann-Kendall
mann_kendall_results_ds = xr.Dataset({'slope': slope, 'p_value': p_value})

# Define o caminho completo para o arquivo NetCDF de saída
output_filename = os.path.join(dir_output, 'mann_kendall_results_focos_sp.nc')

# Salva os resultados em um arquivo NetCDF
mann_kendall_results_ds.to_netcdf(output_filename)

#=========================================================================================================================#
#                                                      PLOTA FIGURA
#=========================================================================================================================#
# cria a moldura da figura figsize=(10,5)
fig, ax = uplt.subplots(axheight=4.5, axwidth=6.8, tight=True, proj='pcarree')

# formatação dos eixos
ax.format(coast=False, borders=False, innerborders=False,
          labels=True, latlines=2, lonlines=2,
          latlim=(latmin_SP+0.5, latmax_SP-0.5), lonlim=(lonmin_SP+0.5, lonmax_SP-0.6),
          small='18px', large='20px',
          title=f'Teste de Mann-Kendall para Focos de Calor \n',
          titleloc='l',
          titleweight='bold',
          titlecolor='bright red')

# plota subtítulo
ax.text(0.000, 1.035,
        f'Satélite: AQUA | Período: 2003-2024 | Resolução: 20km',
        transform=ax.transAxes,
        color='gray',
        fontsize=9,
        verticalalignment='top')

# plota figura do slope com imshow
slope_data_roi = mann_kendall_results_ds['slope'][:,:].salem.roi(shape=shapefile_sp)

lon_coords = mann_kendall_results_ds['lon'].values
lat_coords = mann_kendall_results_ds['lat'].values

# Calculate grid spacing
if len(lon_coords) > 1:
    dx = np.abs(lon_coords[1] - lon_coords[0])
else:
    dx = 0.2 # Default if only one longitude point

if len(lat_coords) > 1:
    dy = np.abs(lat_coords[1] - lat_coords[0])
else:
    dy = 0.2 # Default if only one latitude point

# Calculate extent based on pixel edges
lon_min_extent = lon_coords[0] - dx / 2
lon_max_extent = lon_coords[-1] + dx / 2

# lat_coords is descending, so lat_coords[0] is max_lat, lat_coords[-1] is min_lat
lat_min_extent = lat_coords[-1] - dy / 2
lat_max_extent = lat_coords[0] + dy / 2

extent = [lon_min_extent, lon_max_extent, lat_min_extent, lat_max_extent]

# plota mapa
map1 = ax.imshow(slope_data_roi.values,
                  cmap='coolwarm',
                  extent=extent,
                  transform=ccrs.PlateCarree(),
                  origin='upper',
                  vmin=-1.0, vmax=1.0)

# adiciona o shapefile dos limites de SP
shapefile = list(shpreader.Reader('https://github.com/evmpython/shapefile/raw/main/UFs/SP/SP_UF_2019.shp').geometries())
ax.add_geometries(shapefile, ccrs.PlateCarree(), edgecolor='black', facecolor='none', linewidth=1.5)

# calcula as coordenadas dos pixels com p_value < 0.05 dentro da área de SP
p_value_data_sp = mann_kendall_results_ds['p_value'].salem.roi(shape=shapefile_sp)
significant_mask = (p_value_data_sp < 0.05) & (~np.isnan(p_value_data_sp))

# cria um grid de longitudes e latitudes
lon_grid, lat_grid = np.meshgrid(mann_kendall_results_ds['lon'].values, mann_kendall_results_ds['lat'].values)

# filtra as coordenadas para onde o p_value é significativo
sig_lon_coords = lon_grid[significant_mask.values]
sig_lat_coords = lat_grid[significant_mask.values]

# plota os pontos significativos como bolas pretas e adiciona label para a legenda
ax.scatter(sig_lon_coords, sig_lat_coords, s=25, color='black', marker='o', transform=ccrs.PlateCarree(), zorder=2, label='Significant Trend (p < 0.05)')

# plota barra de cores da figura
cbar = fig.colorbar(map1,
             loc='r',
             label="$\\mathbf{Sen's\\;Slope\\;(Fires/Year\\;Trend)}$",
             ticks=mticker.MultipleLocator(0.25),
             ticklabelsize=11,
             labelsize=13,
             length=0.93,
             width=0.25,
             space=0.3)

# adciona rotação ao nome da barra de cores
cbar.ax.set_ylabel("$\mathbf{Sen's\;Slope\;(Fires/Year\;Trend)}$", rotation=90, labelpad=20)

# labels com 2 casas decimais
cbar.formatter = FormatStrFormatter("%.2f")
cbar.update_ticks()
cbar.ax.minorticks_off()

# sdiciona a legenda ao eixo, posicionando-a levemente acima do canto inferior direito
ax.legend(loc='lower right', bbox_to_anchor=(0.98, 0.015), frameon=True, fancybox=True, shadow=True, borderpad=1, fontsize=10)

# salva figura
fig.savefig(f'{dir_output}/mann_kendall_focos_com_pvalue.jpg', dpi=300, bbox_inches="tight")

In [ ]:
# mostrando os dados de focos
#ds = xr.open_dataset(f'{dir_input}/focos_anuais_brasil_20km_AQUA_2003_2025.nc')
#ds

In [ ]:
# plot rápido
#ds['focos'].sel(time='2025').plot(cmap='jet', vmin=0, vmax=150)

In [ ]:
#mann_kendall_results_ds